# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ssaqlain09/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Refresh / Content Opportunity Scoring.**

I'm picking this lane because I already saw, in Notebook 01, that a learned model beats a hand-written
staleness rule by roughly 3x on Precision@50 (0.740 vs 0.240) on the starter data, and in Notebook 02 that
a simple, readable model can be built and honestly re-checked with a client-holdout split. This lane turns
that same skill (rank pages by evidence, not gut feel) into something a content team could actually act on:
a prioritized review queue instead of a binary "good/bad" label. It also has the deepest supporting data of
the four lanes, per the lane guide's density table (28.9M rows with GSC impressions vs. only 30K AI-session
rows), so I'm not fighting sparse signal from week one.


In [1]:
# Confirms the environment loads before I start reasoning about the lane.
import pandas as pd
print("pandas version:", pd.__version__)


pandas version: 3.0.2


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Question:** Which existing pages should a content reviewer look at first for refresh, expansion,
or protection — given that they can only review a limited number of pages per week?

- **Decision improved:** which pages enter the weekly content-review queue, and in what order.
- **Who acts on it:** a content strategist / SEO editor with limited review capacity (not an automated
  publishing system — a human still decides what to actually change).
- **Unit of analysis:** one content page (one row = one `content_id`, one client).
- **Action someone could take:** refresh stale copy, fix a weak title/meta if CTR is low for its position
  tier, expand thin content, or simply monitor if the drop looks like noise.
- **Cost of a wrong call:**
  - *False positive* (flagged as a priority but not actually a problem): wastes a reviewer's limited hours
    on a page that didn't need attention — the real cost is opportunity cost, since another page that DID
    need help got pushed down the queue.
  - *False negative* (a genuinely declining, high-demand page never gets flagged): the page keeps losing
    visibility/clicks silently until someone notices by accident, which is a slower, harder-to-reverse loss.
  - Because review capacity is scarce, this is a ranking problem, not a classification problem — precision
    in the *top* of the list matters far more than accuracy across the whole dataset.


In [2]:
# No computation needed yet for this section — framing only.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

**What these numbers show:** review capacity is clearly the bottleneck, not lack of candidates — 43.8%
of all pages already show `declining_with_demand`, and roughly a third show a real CTR gap for their
position tier. No reviewer can look at 13,000+ pages by hand. That gap between "how many pages plausibly
need attention" and "how many a human can actually review" is exactly the ranking problem this lane exists
to solve. The very low `stale_visible_page` count (0.1%) also shows staleness alone is a weak signal here —
supporting the case for a learned, multi-signal score over Notebook 01's simple hand rule.


In [3]:
import os
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

stale_visible = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).sum()
declining_with_demand = ((df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)).sum()
low_ctr_visible = ((df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["ctr"] < 0.5)).sum()

print(f"Total pages: {len(df):,}")
print(f"stale_visible_page candidates:      {stale_visible:>6,} ({stale_visible/len(df)*100:.1f}%)")
print(f"declining_with_demand candidates:   {declining_with_demand:>6,} ({declining_with_demand/len(df)*100:.1f}%)")
print(f"low_ctr_visible_page candidates:    {low_ctr_visible:>6,} ({low_ctr_visible/len(df)*100:.1f}%)")


Total pages: 30,000
stale_visible_page candidates:          17 (0.1%)
declining_with_demand candidates:   13,152 (43.8%)
low_ctr_visible_page candidates:     9,759 (32.5%)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim:**
- This work is **observational and decision-support only** — it ranks existing pages by evidence so a
  human reviewer spends limited time on the most promising candidates first.
- Findings are **directional**: e.g. "pages with X pattern were observed to decline more often than pages
  without it, in this dataset, over this time window."
- A ranked queue with reason codes a reviewer can inspect and override.

**What I will never claim:**
- That a refresh *caused* a recovery — proving causation needs an actual experiment (e.g. before/after
  test on similar pages), which this data alone cannot give me.
- That I've reverse-engineered a Google ranking factor — I only have FlyRank's observed search/engagement
  signals, never the algorithm itself.
- That the model score is "the truth" — it's a prioritization tool. A page scored high can still turn out,
  on human review, to have a fine explanation (consolidation, seasonality) that the model didn't capture.
- Any claim about a specific client, URL, or query — everything in this dataset is pseudonymized, and my
  write-up will stay pseudonymized too, per the lane guide's public-safe output rules.


In [4]:
# No computation needed for this section — framing only.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.